# PGA Tour Player Comparison (2015–2022)
Data: [PGA Tour Golf Data 2015–2022](https://www.kaggle.com/datasets/robikscube/pga-tour-golf-data-20152022) on Kaggle. One row per player per tournament.

In [ ]:
%pip install kagglehub ipywidgets matplotlib pandas

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

## Load the data
Tries Kaggle first. If that doesn't work, it uses a public GitHub copy of the same CSV.

In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download("robikscube/pga-tour-golf-data-20152022")
    csv_file = glob.glob(os.path.join(path, "**", "*Tourn Level*.csv"), recursive=True)[0]
    df = pd.read_csv(csv_file)
except Exception as e:
    print("Kaggle download didn't work, using GitHub copy instead:", e)
    url = ("https://raw.githubusercontent.com/jordanr-johnson/PGA-Tournament-Python-Project/"
           "main/ASA%20All%20PGA%20Raw%20Data%20-%20Tourn%20Level.csv")
    df = pd.read_csv(url)

print(df.shape)

## Similar players
Pick a player to see who they compare to. Similarity uses play style (strokes gained off the tee, approach, around the green, putting), scoring (score to par per round) and results (cut % and top-10 %). Every stat is put on the same scale, and the players with the smallest overall difference rank highest. Only players with 20+ shot-tracked events are included.

In [ ]:
sg_cols = ["sg_ott", "sg_app", "sg_arg", "sg_putt"]
d = df.dropna(subset=sg_cols).copy()
d["to_par"] = d["strokes"] - d["hole_par"]
d["top10"] = d["pos"] <= 10

def summarize(g):
    w = g["n_rounds"]
    row = {c: (g[c] * w).sum() / w.sum() for c in sg_cols}
    row["score_per_round"] = g["to_par"].sum() / w.sum()
    row["cut_pct"] = g["made_cut"].mean() * 100
    row["top10_pct"] = g["top10"].mean() * 100
    row["events"] = len(g)
    return pd.Series(row)

players = d.groupby("player").apply(summarize)
players = players[players["events"] >= 20]
print("Players with 20+ tracked events:", len(players))

features = sg_cols + ["score_per_round", "cut_pct", "top10_pct"]
z = (players[features] - players[features].mean()) / players[features].std()
sg_labels = ["Off Tee", "Approach", "Around Green", "Putting"]

def show_similar(player, n_similar=10):
    dist = np.sqrt(((z - z.loc[player]) ** 2).sum(axis=1))
    sim_all = 100 * (1 - dist / dist.max())
    result = players.assign(similarity=sim_all).sort_values("similarity", ascending=False).head(n_similar + 1)
    names = result.index.tolist()
    n = len(names)

    fig, (ax1, ax2, ax3) = plt.subplots(
        1, 3, figsize=(14, 0.5 * n + 1.8), sharey=True,
        gridspec_kw={"width_ratios": [2, 3, 2.4]})

    sim = result["similarity"].values
    ax1.barh(range(n), sim, color=["gray"] + ["tab:blue"] * (n - 1))
    for i, s in enumerate(sim):
        ax1.text(s - 1, i, "selected" if i == 0 else f"{s:.0f}%", va="center", ha="right", color="white", fontsize=9)
    ax1.set_yticks(range(n))
    ax1.set_yticklabels(names)
    ax1.invert_yaxis()
    ax1.set_xlim(0, 100)
    ax1.set_title("Similarity")
    ax1.spines[["top", "right"]].set_visible(False)

    sg_vals = result[sg_cols].values
    lim = np.abs(sg_vals).max()
    ax2.imshow(sg_vals, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
    for i in range(n):
        for j in range(4):
            v = sg_vals[i, j]
            ax2.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=9,
                     color="white" if abs(v) > 0.6 * lim else "black")
    ax2.set_xticks(range(4))
    ax2.set_xticklabels(sg_labels)
    ax2.xaxis.tick_top()
    ax2.set_title("Play style: strokes gained per round\n(blue = better than tour avg, red = worse)", pad=28)

    cols = [("score_per_round", "Score/Rd", "{:+.2f}"), ("cut_pct", "Cut %", "{:.0f}%"),
            ("top10_pct", "Top 10 %", "{:.0f}%"), ("events", "Events", "{:.0f}")]
    for j, (c, lab, fmt) in enumerate(cols):
        for i in range(n):
            ax3.text(j, i, fmt.format(result[c].iloc[i]), ha="center", va="center", fontsize=10)
        ax3.text(j, -0.9, lab, ha="center", va="bottom", fontsize=10)
    ax3.set_xlim(-0.5, len(cols) - 0.5)
    ax3.set_title("Scoring & results", pad=28)
    ax3.axis("off")

    fig.suptitle(f"Players most similar to {player} (2015-2022, events with strokes gained data)", fontsize=13)
    plt.tight_layout()
    plt.show()

player_list = sorted(players.index)
interact(show_similar,
         player=widgets.Dropdown(options=player_list, value="Collin Morikawa", description="Player:"),
         n_similar=widgets.IntSlider(value=10, min=5, max=20, description="# similar:"));

## Strokes gained vs. tour average
Where is this player above or below the tour average? Strokes gained is already measured against the field, so 0 is the tour average. Rows with no strokes gained data (mostly tournaments without shot tracking) are left out, and averages are weighted by rounds played.

In [ ]:
sg_all = ["sg_ott", "sg_app", "sg_arg", "sg_putt", "sg_total"]
sg_data = df.dropna(subset=sg_all)
print("Rows dropped for missing strokes gained:", len(df) - len(sg_data))

def show_sg(player):
    p = sg_data[sg_data["player"] == player]
    values = [(p[c] * p["n_rounds"]).sum() / p["n_rounds"].sum() for c in sg_all]
    labels = ["Off the Tee", "Approach", "Around the Green", "Putting", "Total"]
    colors = ["tab:blue" if v >= 0 else "tab:red" for v in values]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels, values, color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.invert_yaxis()

    for i, v in enumerate(values):
        ax.text(v + (0.02 if v >= 0 else -0.02), i, f"{v:+.2f}",
                va="center", ha="left" if v >= 0 else "right")

    ax.set_xlim(min(values + [0]) - 0.3, max(values) + 0.3)
    ax.set_xlabel("Strokes gained per round (0 = tour average)")
    ax.set_title(f"{player}: Strokes Gained vs. Tour Average ({len(p)} events, 2015-2022)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

interact(show_sg,
         player=widgets.Dropdown(options=player_list, value="Collin Morikawa", description="Player:"));